In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Lasso, Ridge
from sklearn.decomposition import PCA
from sklearn.metrics import r2_score, mean_squared_error

In [4]:
# Step 1: Load and Clean Data
df = pd.read_csv('diamonds.csv')
df = df.drop('Unnamed: 0', axis=1)  # Drop index column
df = df.drop_duplicates()  # Remove duplicates
df = df[(df['x'] > 0) & (df['y'] > 0) & (df['z'] > 0)]  # Valid dimensions

In [5]:
# Remove price outliers (IQR method)
Q1 = df['price'].quantile(0.25)
Q3 = df['price'].quantile(0.75)
IQR = Q3 - Q1
lower, upper = Q1 - 1.5*IQR, Q3 + 1.5*IQR
df = df[(df['price'] >= lower) & (df['price'] <= upper)]
print(f"Cleaned shape: {df.shape}")


Cleaned shape: (50255, 10)


In [6]:
# EDA (correlations, distributions)
cont_cols = ['carat', 'depth', 'table', 'x', 'y', 'z']
print("Correlations with price:\n", df[cont_cols + ['price']].corr()['price'].sort_values(ascending=False))

Correlations with price:
 price    1.000000
carat    0.915838
x        0.897440
y        0.892718
z        0.873020
table    0.127453
depth    0.004179
Name: price, dtype: float64


In [7]:
# Step 2: Create diamonds_model sample
diamonds_model = df.sample(n=12500, random_state=42)
print(f"Sample shape: {diamonds_model.shape}")

Sample shape: (12500, 10)


In [9]:
# Step 3: Prepare features for modeling
X = diamonds_model.drop('price', axis=1)
y = diamonds_model['price']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

cat_cols = ['cut', 'color', 'clarity']
num_cols = ['carat', 'depth', 'table', 'x', 'y', 'z']

In [10]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(drop='first', sparse_output=False), cat_cols)
    ])

In [11]:
# Model 3: Linear Regression (all features)
lr_pipeline = Pipeline([('preprocessor', preprocessor), ('regressor', LinearRegression())])
lr_pipeline.fit(X_train, y_train)
y_pred_lr = lr_pipeline.predict(X_test)
r2_lr = r2_score(y_test, y_pred_lr)
rmse_lr = np.sqrt(mean_squared_error(y_test, y_pred_lr))
print(f"LR Full - R²: {r2_lr:.4f}, RMSE: {rmse_lr:.2f}")

LR Full - R²: 0.8889, RMSE: 928.76


In [12]:
# Model 4: PCA on continuous + LR
num_pipeline = Pipeline([('scaler', StandardScaler()), ('pca', PCA(n_components=2))])
num_transformed = num_pipeline.fit_transform(X[num_cols])
pca_df = pd.DataFrame(num_transformed, columns=['PC1', 'PC2'])
pca_df['price'] = y
X_pca = pca_df[['PC1', 'PC2']]
X_train_pca, X_test_pca, y_train_pca, y_test_pca = train_test_split(X_pca, y, test_size=0.2, random_state=42)
pca_scaler = StandardScaler()
X_train_pca_s = pca_scaler.fit_transform(X_train_pca)
X_test_pca_s = pca_scaler.transform(X_test_pca)
lr_pca = LinearRegression().fit(X_train_pca_s, y_train_pca)
y_pred_pca = lr_pca.predict(X_test_pca_s)
r2_pca = r2_score(y_test_pca, y_pred_pca)
rmse_pca = np.sqrt(mean_squared_error(y_test_pca, y_pred_pca))
print(f"LR PCA - R²: {r2_pca:.4f}, RMSE: {rmse_pca:.2f}")
print(f"PCA variance explained: {num_pipeline[1].explained_variance_ratio_.sum():.4f}")

LR PCA - R²: 0.8088, RMSE: 1218.34
PCA variance explained: 0.8782


In [13]:
# Models 5: Lasso & Ridge with alpha tuning
alphas = [0.1, 1.0, 10.0]

# Lasso
best_r2_lasso, best_rmse_lasso, best_a_l = 0, np.inf, 0
for a in alphas:
    lasso = Pipeline([('preprocessor', preprocessor), ('regressor', Lasso(alpha=a, random_state=42, max_iter=2000))])
    lasso.fit(X_train, y_train)
    y_pred_l = lasso.predict(X_test)
    r2_l = r2_score(y_test, y_pred_l)
    rmse_l = np.sqrt(mean_squared_error(y_test, y_pred_l))
    if r2_l > best_r2_lasso:
        best_r2_lasso, best_rmse_lasso, best_a_l = r2_l, rmse_l, a
print(f"Lasso (α={best_a_l}) - R²: {best_r2_lasso:.4f}, RMSE: {best_rmse_lasso:.2f}")

Lasso (α=1.0) - R²: 0.9162, RMSE: 806.74


In [14]:
# Ridge
best_r2_ridge, best_rmse_ridge, best_a_r = 0, np.inf, 0
for a in alphas:
    ridge = Pipeline([('preprocessor', preprocessor), ('regressor', Ridge(alpha=a, random_state=42))])
    ridge.fit(X_train, y_train)
    y_pred_r = ridge.predict(X_test)
    r2_r = r2_score(y_test, y_pred_r)
    rmse_r = np.sqrt(mean_squared_error(y_test, y_pred_r))
    if r2_r > best_r2_ridge:
        best_r2_ridge, best_rmse_ridge, best_a_r = r2_r, rmse_r, a
print(f"Ridge (α={best_a_r}) - R²: {best_r2_ridge:.4f}, RMSE: {best_rmse_ridge:.2f}")

Ridge (α=10.0) - R²: 0.9018, RMSE: 873.01


In [16]:
# Comparison table
results = pd.DataFrame({
    'Model': ['LR Full', 'LR PCA', 'Lasso', 'Ridge'],
    'R2': [r2_lr, r2_pca, best_r2_lasso, best_r2_ridge],
    'RMSE': [rmse_lr, rmse_pca, best_rmse_lasso, best_rmse_ridge]})

In [17]:
print(results)
best_model = results.loc[results['R2'].idxmax()]
print(f"\nBest: {best_model['Model']} (R²={best_model['R2']:.4f})")

     Model        R2         RMSE
0  LR Full  0.888907   928.762407
1   LR PCA  0.808832  1218.340086
2    Lasso  0.916180   806.743366
3    Ridge  0.901844   873.011988

Best: Lasso (R²=0.9162)
